# Ch 06. パーセプトロンからMLPへ — 解答

> このノートブックは5問すべての厳密な解答と再現可能な検証を含む。

## 問題 1

**問題**: NAND 関数 学習, 出力.

### 厳密な解説

制御する要因: **training examples and labels (fixed NAND truth table)**。  測定指標: **learned weights, bias, and four predictions**。  解釈と限界: The perceptron update changes parameters only on mistakes. NAND is linearly separable, so the finite pass converges; the exact parameters depend on presentation order.


In [ ]:
import torch
X=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=torch.tensor([1.,1.,1.,0.]); w=torch.zeros(2); b=torch.tensor(0.)
for _ in range(20):
 for xi,yi in zip(X,y):
  e=yi-(xi@w+b>=0).float(); w+=e*xi; b+=e
p=(X@w+b>=0).int(); print({'weights':w.tolist(),'bias':b.item(),'predictions':p.tolist()}); assert torch.equal(p,y.int())


## 問題 2

**問題**: 2, 4, 8, 16 XOR 学習 速度 比較.

### 厳密な解説

制御する要因: **hidden width: 2, 4, 8, 16**。  測定指標: **steps to 100% XOR accuracy and final loss**。  解釈と限界: All widths receive the same XOR samples, optimizer, initialization rule, and step budget. Width affects optimization capacity, but a single tiny run is not a universal ranking.


In [ ]:
import torch
X=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]]); y=torch.tensor([0,1,1,0])
for h in (2,4,8,16):
 torch.manual_seed(60); m=torch.nn.Sequential(torch.nn.Linear(2,h),torch.nn.Tanh(),torch.nn.Linear(h,2)); o=torch.optim.Adam(m.parameters(),lr=.05); reached=None
 for step in range(1,401):
  loss=torch.nn.functional.cross_entropy(m(X),y); o.zero_grad(); loss.backward(); o.step()
  if reached is None and (m(X).argmax(1)==y).all(): reached=step
 print({'width':h,'steps_to_full_accuracy':reached,'final_loss':round(loss.item(),6)})


## 問題 3

**問題**: MNIST MLP 0( ), 1, 2 複雑度 比較.

### 厳密な解説

制御する要因: **number of hidden layers: 0, 1, 2**。  測定指標: **classification accuracy on a fixed nonlinear synthetic test set**。  解釈と限界: A two-moons-style analytic dataset replaces MNIST to keep the experiment offline. It tests the requested depth effect, not a claimed MNIST score.


In [ ]:
import torch
torch.manual_seed(61); X=torch.rand(400,2)*2-1; y=((X[:,0]**2+X[:,1]**2)>.45).long(); tr=torch.arange(300); te=torch.arange(300,400)
for depth in (0,1,2):
 layers=[]; d=2
 for _ in range(depth): layers += [torch.nn.Linear(d,16),torch.nn.ReLU()]; d=16
 layers += [torch.nn.Linear(d,2)]; m=torch.nn.Sequential(*layers); o=torch.optim.Adam(m.parameters(),lr=.03)
 for _ in range(150):
  loss=torch.nn.functional.cross_entropy(m(X[tr]),y[tr]); o.zero_grad(); loss.backward(); o.step()
 print({'hidden_layers':depth,'test_accuracy':round((m(X[te]).argmax(1)==y[te]).float().mean().item(),3)})


## 問題 4

**問題**: ReLU sigmoid MNIST 学習 .

### 厳密な解説

制御する要因: **activation: ReLU versus sigmoid**。  測定指標: **final loss and first-layer gradient norm**。  解釈と限界: Matched initial weights and data isolate the activation. Sigmoid saturation can shrink gradients; this reduced synthetic task demonstrates the mechanism rather than fabricating MNIST results.


In [ ]:
import torch
torch.manual_seed(62); X=torch.randn(128,16)*3; y=(X[:,:3].sum(1)>0).long(); base=torch.nn.Linear(16,32); out=torch.nn.Linear(32,2)
for name,act in [('relu',torch.relu),('sigmoid',torch.sigmoid)]:
 h=torch.nn.Linear(16,32); q=torch.nn.Linear(32,2); h.load_state_dict(base.state_dict()); q.load_state_dict(out.state_dict()); o=torch.optim.SGD([*h.parameters(),*q.parameters()],lr=.1)
 for _ in range(60): loss=torch.nn.functional.cross_entropy(q(act(h(X))),y); o.zero_grad(); loss.backward(); o.step()
 print({'activation':name,'loss':round(loss.item(),5),'first_layer_grad_norm':round(h.weight.grad.norm().item(),5)})


## 問題 5

**問題**: " " .

### 厳密な解説

制御する要因: **number of localized ReLU hinge units**。  測定指標: **maximum approximation error for |x|**。  解釈と限界: A one-hidden-layer ReLU network represents a piecewise-linear function by summing hinges: |x| = ReLU(x)+ReLU(-x). Width, not additional depth, supplies more breakpoints; this illustrates the theorem but is not its full proof.


In [ ]:
import torch
x=torch.linspace(-2,2,401); approx=torch.relu(x)+torch.relu(-x); err=(approx-x.abs()).abs().max().item(); print({'hidden_units':2,'max_abs_error':err}); assert err==0
